In [ ]:
import scanpy as sc
import scvi

Read anndata object

In [ ]:
adata = sc.read_h5ad("c:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/AnndataR/AnnDataR.h5ad")
adata

In [ ]:
adata.obs["orig.ident"].unique()

In [ ]:
# this is the column which needs to be specified in the model training
adata.obs["experiment"].unique()

In [ ]:
adata.obs["experiment"] = adata.obs["experiment"].replace(
    "CITEseq_LNP_pIC_LNPS",
    "CITEseq_LNP_pIC_LNPs"
)

In [ ]:
adata.obs["experiment"].unique()

In [ ]:
# integration process is stochastic, so set seed
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

In [ ]:
# for test runs, we subset the object to make runtime easier and less expensive
# first 4000 cells are used here 
# adata = adata[:4000, :].copy()

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=3000,
    batch_key="orig.ident",  # or your chosen batch key
    flavor="seurat_v3" # can be confusing that the data was not scaled or normalized before this step, but Seurat v3 expects raw counts
)

Since all 36000 genes is a lot to train on, we will only use HVG

In [ ]:
adata_hvg = adata[:, adata.var["highly_variable"]].copy()

Integration with scVI (from tutorial)

In [ ]:
# create an SCVI model object for training. The key identifies the sample and the layer key with the counts
# the different experiments have different treatments, so there is different biological variability, yet, we are only interested in the cell type stages
# so biological variability is a confounding factor, which we want to remove. Therefore, we specify the batch_key as the column in the adata.obs which contains the different experiments
scvi.model.SCVI.setup_anndata(adata_hvg, 
                              layer=None, 
                              batch_key="orig.ident")

In [ ]:
# setup model
model = scvi.model.SCVI(adata_hvg, n_layers=2, n_latent=30, gene_likelihood="nb")

In [ ]:
# train model
model.train(max_epochs=1000, 
            early_stopping=True, 
            accelerator = "cpu", # torch package is incompatible with the GPU, so an older version needs to be installed to use the GPU. For now, we will use the CPU for training
            early_stopping_patience=10, 
            plan_kwargs={"lr": 1e-3})

# epochs is the amount of training iterations, early stopping stops the training in case of convergence
# early stopping patience is the number of epochs to wait before stopping the training if no improvement is seen
# plan_kwargs is the learning rate, which is the step size for updating the model parameters during training. A smaller learning rate can lead to more stable training but may require more epochs to converge, while a larger learning rate can speed up training but may lead to instability or divergence.

In [ ]:
# save model to be safe
model.save("scvi_model_RNA_integration")

Evaluate the model, now that it is trained

In [ ]:
# extract the latent representation of the space
SCVI_LATENT_KEY = "X_scVI"
adata.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

In [ ]:
# calculate these metrics for the latent representation
# here, we use the latent coordinates, not PCA coordinates for the neighbourhood graph
sc.pp.neighbors(adata, use_rep=SCVI_LATENT_KEY)
sc.tl.leiden(
    adata,
    resolution=1.0,
    key_added="leiden_1"
)
sc.tl.umap(adata)

In [ ]:
sc.pl.umap(adata,
           color=["orig.ident", "celltype_new", "experiment", "leiden_1", "treatment"],
           frameon=False,
           legend_loc="on data",
           ncols=10)

In [ ]:
sc.pl.umap(adata,
           color=["orig.ident", "celltype_new", "experiment", "leiden_1", "treatment"],
           frameon=False,
           ncols=10)

In [ ]:
# because of a module incompability, we cannot save the model itself, so we will save the anndata object with the latent representation, which can be used for downstream analysis
sc.write("C:/Users/irc/Desktop/Internship Bioinformatics 2025-2026/Lode/Integration_scVI/scVI_integrated_object", adata)

Here, we make a for loop to plot certain features on separate groups

In [ ]:
adata_jve010 = adata[adata.obs["orig.ident"] == "JVE010"].copy()

# Marker dictionary
markers = {
    "Pre_cDC1": ["Ccr2", "Fcer1g", "Cd24a", "Vim"],
    "Early_Immature": ["Sell", "Creld2", "Pdia4"],
    "Later_Immature": ["Cd207", "Itgae", "Apol7c", "Apoe", "Dnase1l3", "Cadm1", "Xcr1", "Cd83", "Cd86"],
    "Early_Mature": ["Cxcl10", "Cxcl9", "Iigp1", "Ifi47", "Gbp2", "Gbp5", "Cd40"],
    "Late_Mature": ["Cd63", "Fscn1", "Il4i1", "Socs2", "Ccr7"]
}

# Plot one figure per marker set
for stage, genes in markers.items():

    # Only keep genes present in the dataset
    genes_present = [g for g in genes if g in adata_jve010.var_names]

    print(f"{stage}: {len(genes_present)}/{len(genes)} genes found")

    if len(genes_present) > 0:
        sc.pl.umap(
            adata_jve010,
            color=genes_present,
            title=[f"{stage}: {g}" for g in genes_present],
            ncols=3
        )

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# all orig.ident groups
orig_groups = adata.obs["orig.ident"].unique()

# flatten marker dictionary
genes = [gene for stage in markers.values() for gene in stage]

# keep only genes present
genes = [g for g in genes if g in adata.var_names]

for gene in genes:

    print(f"Plotting {gene}")

    fig, axes = plt.subplots(
        1,
        len(orig_groups),
        figsize=(5 * len(orig_groups), 5)
    )

    # if only one subset exists
    if len(orig_groups) == 1:
        axes = [axes]

    for ax, orig in zip(axes, orig_groups):

        adata_subset = adata[
            adata.obs["orig.ident"] == orig
        ]

        sc.pl.umap(
            adata_subset,
            color=gene,
            title=f"{orig}: {gene}",
            ax=ax,
            show=False
        )

    plt.tight_layout()
    plt.show()